# Food Review Classification — Full Pipeline

รันครบใน notebook เดียว: **เทรน Baseline + XLM-R → Evaluate → Visualize → Score**

**ก่อนรัน:** เปิด notebook จากโฟลเดอร์โปรเจกต์ (ที่มี `config.py`) และติดตั้ง dependencies

```bash
pip install -r requirements.txt
pip install jupyter
```

ข้อมูล Wongnai: วาง `data/wongnai/train.csv` + `test.csv` หรือรัน `python scripts/download_wongnai.py`

## 1. Setup

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import IFrame, display

# โฟลเดอร์โปรเจกต์ (ต้องมี config.py)
ROOT = Path.cwd()
if not (ROOT / "config.py").exists():
    candidate = ROOT / "foods-review-classification"
    if (candidate / "config.py").exists():
        ROOT = candidate
    else:
        raise FileNotFoundError(
            "ไม่พบ config.py — chdir ไปที่ root ของโปรเจกต์ก่อนรัน notebook"
        )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import config

print(f"Project root: {ROOT}")
print(f"Train data:   {config.RAW_DATA_PATH}")
print(f"Test data:    {config.WONGNAI_TEST_PATH}")
print(f"PyTorch device: {config.TORCH_DEVICE}")

## 2. ตั้งค่าการรัน (แก้ได้ตรงนี้)

In [ ]:
# ขั้นตอนที่จะรัน
RUN_TRAIN_BASELINE = True
RUN_TRAIN_XLMR = True
RUN_EVALUATE = True
RUN_VISUALIZE = True
RUN_SCORE = True

# ข้ามเทรนถ้ามี artifact อยู่แล้ว (เร็วขึ้นตอนรันซ้ำ)
SKIP_TRAIN_IF_ARTIFACTS_EXIST = True

# ชุดทดสอบสำหรับ evaluate / score
TEST_CSV = config.WONGNAI_TEST_PATH
EVAL_JSON = config.DEFAULT_EVAL_REPORT
EVAL_HTML = config.DEFAULT_EVAL_VIZ
SCORED_CSV = config.DEFAULT_SCORED_OUTPUT

## 3. ตรวจสอบข้อมูล

In [ ]:
def require_file(path: str, label: str) -> None:
    if not Path(path).is_file():
        raise FileNotFoundError(
            f"{label} ไม่พบ: {path}\n"
            "  Wongnai: python scripts/download_wongnai.py\n"
            "  Mock:    python scripts/generate_mock_data.py"
        )

require_file(config.RAW_DATA_PATH, "Train CSV")
require_file(TEST_CSV, "Test CSV")

df_train_peek = pd.read_csv(config.RAW_DATA_PATH, nrows=3)
df_test_peek = pd.read_csv(TEST_CSV, nrows=3)
print(f"Train rows (file): {sum(1 for _ in open(config.RAW_DATA_PATH, encoding='utf-8')) - 1}")
print(f"Test rows (file):  {sum(1 for _ in open(TEST_CSV, encoding='utf-8')) - 1}")
display(df_train_peek)
display(df_test_peek)

## 4. เทรน Baseline (TF-IDF + XGBoost)

In [ ]:
baseline_ready = (
    Path(config.TFIDF_VECTORIZER_PATH).is_file()
    and Path(config.XGB_MODEL_PATH).is_file()
)

if RUN_TRAIN_BASELINE:
    if SKIP_TRAIN_IF_ARTIFACTS_EXIST and baseline_ready:
        print("ข้าม train_baseline — พบ artifact แล้ว")
    else:
        import train_baseline

        train_baseline.main()
else:
    print("ข้าม RUN_TRAIN_BASELINE")

## 5. เทรน XLM-R (PyTorch manual loop)

In [ ]:
xlmr_ready = (Path(config.XLMR_ARTIFACTS_DIR) / "config.json").is_file()

if RUN_TRAIN_XLMR:
    if SKIP_TRAIN_IF_ARTIFACTS_EXIST and xlmr_ready:
        print("ข้าม train_xlmr — พบ artifact แล้ว")
    else:
        import train_xlmr

        train_xlmr.main()
else:
    print("ข้าม RUN_TRAIN_XLMR")

## 6. Evaluate (เปรียบเทียบ Baseline vs XLM-R)

In [ ]:
if RUN_EVALUATE:
    import utils
    from evaluate import (
        ensure_artifacts,
        evaluate_model,
        predict_baseline,
        predict_xlmr,
        print_comparison,
    )

    ensure_artifacts("both")
    df_test = utils.load_and_standardize_data(TEST_CSV)

    results = {"input": TEST_CSV, "models": {}}

    exp_b = predict_baseline(df_test)
    results["models"]["baseline"] = evaluate_model(
        "Baseline (TF-IDF + XGBoost)", df_test, exp_b
    )

    exp_x = predict_xlmr(df_test)
    results["models"]["xlmr"] = evaluate_model("XLM-R", df_test, exp_x)

    print_comparison(results["models"]["baseline"], results["models"]["xlmr"])

    Path(EVAL_JSON).parent.mkdir(parents=True, exist_ok=True)
    with open(EVAL_JSON, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"\nSaved: {EVAL_JSON}")

    summary = pd.DataFrame(
        {
            "metric": ["MAE", "RMSE", "Accuracy", "F1 macro"],
            "baseline": [
                results["models"]["baseline"]["mae"],
                results["models"]["baseline"]["rmse"],
                results["models"]["baseline"]["accuracy"],
                results["models"]["baseline"]["f1_macro"],
            ],
            "xlmr": [
                results["models"]["xlmr"]["mae"],
                results["models"]["xlmr"]["rmse"],
                results["models"]["xlmr"]["accuracy"],
                results["models"]["xlmr"]["f1_macro"],
            ],
        }
    )
    display(summary.style.format(precision=4))
else:
    print("ข้าม RUN_EVALUATE")

## 7. Visualize (HTML dashboard)

In [ ]:
if RUN_VISUALIZE:
    if not Path(EVAL_JSON).is_file():
        raise FileNotFoundError(f"ไม่พบ {EVAL_JSON} — รัน Evaluate ก่อน")

    from visualize_eval import load_report, render_html

    report = load_report(EVAL_JSON)
    html = render_html(report, EVAL_JSON)
    Path(EVAL_HTML).parent.mkdir(parents=True, exist_ok=True)
    with open(EVAL_HTML, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"Saved: {EVAL_HTML}")
    print("แสดงตัวอย่างด้านล่าง (หรือเปิดไฟล์ในเบราว์เซอร์)")
    rel_html = Path(EVAL_HTML).relative_to(ROOT).as_posix()
    display(IFrame(src=rel_html, width="100%", height=720))
else:
    print("ข้าม RUN_VISUALIZE")

## 8. Score (ทำนาย + anomaly บนชุด test)

In [ ]:
if RUN_SCORE:
    import numpy as np
    import utils
    from score import get_hex_color, predict_xlmr

    df_score = utils.load_and_standardize_data(TEST_CSV)
    df_score["ai_expected_rating"] = predict_xlmr(df_score)
    df_score["ai_hex_color"] = df_score["ai_expected_rating"].apply(get_hex_color)
    df_score["delta"] = np.abs(df_score["user_rating"] - df_score["ai_expected_rating"])
    df_score["is_anomaly"] = df_score["delta"] >= config.ANOMALY_THRESHOLD

    Path(SCORED_CSV).parent.mkdir(parents=True, exist_ok=True)
    df_score.to_csv(SCORED_CSV, index=False)
    print(f"Saved: {SCORED_CSV}")
    print(f"Anomalies (delta >= {config.ANOMALY_THRESHOLD}): {df_score['is_anomaly'].sum()} / {len(df_score)}")
    display(df_score.head(10))
else:
    print("ข้าม RUN_SCORE")

## 9. สรุปผลลัพธ์

In [ ]:
outputs = {
    "Baseline artifacts": config.BASELINE_ARTIFACTS_DIR,
    "XLM-R artifacts": config.XLMR_ARTIFACTS_DIR,
    "Eval JSON": EVAL_JSON if Path(EVAL_JSON).exists() else "(ยังไม่สร้าง)",
    "Eval HTML": EVAL_HTML if Path(EVAL_HTML).exists() else "(ยังไม่สร้าง)",
    "Scored CSV": SCORED_CSV if Path(SCORED_CSV).exists() else "(ยังไม่สร้าง)",
}
for label, path in outputs.items():
    print(f"{label:20} → {path}")

print("\nPipeline เสร็จแล้ว")